In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

# ============================================================
# 1. LOAD DATASET
# ============================================================

df = pd.read_csv("/content/placement_predict_50k_adjusted (1).csv")

print("Dataset Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())


# ============================================================
# 2. TARGET VARIABLE
# ============================================================

# PlacementStatus is already:
# 0 = Not Placed
# 1 = Placed

y = df["PlacementStatus"]


# ============================================================
# 3. SELECT PREDICTOR VARIABLES
# ============================================================

features = [
    "Gender",
    "City",
    "CollegeTier",
    "Stream",
    "Specialisation",
    "Hostel",
    "HistoryOfBacklogs",

    #"SGPA_Sem1", # Removed as not in dataset
    #"SGPA_Sem2", # Removed as not in dataset
    #"SGPA_Sem3", # Removed as not in dataset
    #"SGPA_Sem4", # Removed as not in dataset
    #"SGPA_Sem5", # Removed as not in dataset
    #"SGPA_Sem6", # Removed as not in dataset
    #"SGPA_Sem7", # Removed as not in dataset
    #"SGPA_Sem8", # Removed as not in dataset

    "CGPA",
    "AttendancePercent",

    "Internships",
    "Projects",
    "Workshops",
    "Certifications",
    "Publications",

    "AptitudeTestScore",
    "SoftSkillsRating",
    "CodingTestScore",
    "MockInterviewScore",
    "ExtraCurricular"
]

#X = df[features].copy()
X = df[features]

# ============================================================
# 4. CATEGORICAL VARIABLES
# ============================================================

categorical_features = [
    "Gender",
    "City",
    "CollegeTier",
    "Stream",
    "Specialisation",
    "Hostel",
    "HistoryOfBacklogs",
    "ExtraCurricular" # Moved from numerical_features
]


# ============================================================
# 5. NUMERICAL VARIABLES
# ============================================================

numerical_features = [
    #"SGPA_Sem1", # Removed as not in dataset
    #"SGPA_Sem2", # Removed as not in dataset
    #"SGPA_Sem3", # Removed as not in dataset
    #"SGPA_Sem4", # Removed as not in dataset
    #"SGPA_Sem5", # Removed as not in dataset
    #"SGPA_Sem6", # Removed as not in dataset
    #"SGPA_Sem7", # Removed as not in dataset
    #"SGPA_Sem8", # Removed as not in dataset

    "CGPA",
    "AttendancePercent",

    "Internships",
    "Projects",
    "Workshops",
    "Certifications",
    "Publications",

    "AptitudeTestScore",
    "SoftSkillsRating",
    "CodingTestScore",
    "MockInterviewScore"
    #"ExtraCurricular" # Moved to categorical_features
]


# ============================================================
# 6. HANDLE MISSING VALUES
# ============================================================

for col in numerical_features:
    X[col] = X[col].fillna(X[col].median())

for col in categorical_features:
    X[col] = X[col].fillna(X[col].mode()[0])


# ============================================================
# 7. TRAIN-TEST SPLIT
# ============================================================
#stratify=y = Keep the same proportion of values in both training and testing splits.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


# ============================================================
# 8. PREPROCESSING
# ============================================================
#ColumnTransformer allows us to preprocess different types of features differently and combine the results into a single feature matrix

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),

        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            ),
            categorical_features
        )
    ]
)


# ============================================================
# 9. BINOMIAL LOGISTIC REGRESSION
# ============================================================
#The purpose of Pipeline is to connect multiple machine-learning steps into a single sequential workflow.

logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),

        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                random_state=42
            )
        )
    ]
)


# ============================================================
# 10. TRAIN MODEL
# ============================================================

logistic_model.fit(X_train, y_train)


# ============================================================
# 11. PREDICTION
# ============================================================

y_pred = logistic_model.predict(X_test)

# Probability of Placement = class 1
y_probability = logistic_model.predict_proba(X_test)[:, 1]


# ============================================================
# 12. MODEL EVALUATION
# ============================================================

accuracy = accuracy_score(y_test, y_pred)

print("\n============================================")
print("BINOMIAL LOGISTIC REGRESSION RESULTS")
print("============================================")

print("\nAccuracy:")
print(round(accuracy, 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Not Placed", "Placed"]
    )
)

print("\nROC-AUC:")
print(round(roc_auc_score(y_test, y_probability), 4))

Dataset Shape: (50000, 21)

Columns:
['Gender', 'City', 'CollegeTier', 'Stream', 'Specialisation', 'Hostel', 'HistoryOfBacklogs', 'CGPA', 'AttendancePercent', 'Internships', 'Projects', 'Workshops', 'Certifications', 'Publications', 'AptitudeTestScore', 'SoftSkillsRating', 'CodingTestScore', 'MockInterviewScore', 'ExtraCurricular', 'PlacementStatus', 'IsAnomaly']


/tmp/ipykernel_2885/573745585.py:132: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[col] = X[col].fillna(X[col].median())
/tmp/ipykernel_2885/573745585.py:135: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[col] = X[col].fillna(X[col].mode()[0])



BINOMIAL LOGISTIC REGRESSION RESULTS

Accuracy:
0.7899

Confusion Matrix:
[[4228 1022]
 [1079 3671]]

Classification Report:
              precision    recall  f1-score   support

  Not Placed       0.80      0.81      0.80      5250
      Placed       0.78      0.77      0.78      4750

    accuracy                           0.79     10000
   macro avg       0.79      0.79      0.79     10000
weighted avg       0.79      0.79      0.79     10000


ROC-AUC:
0.8769


In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

# ============================================================
# 1. LOAD DATASET
# ============================================================

df = pd.read_csv("/content/placement_predict_50k_adjusted (1).csv")

print("Dataset Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())


# ============================================================
# 2. TARGET VARIABLE
# ============================================================

# PlacementStatus is already:
# 0 = Not Placed
# 1 = Placed

y = df["PlacementStatus"]


# ============================================================
# 3. SELECT PREDICTOR VARIABLES
# ============================================================

features = [
    "Gender",
    "City",
    "CollegeTier",
    "Stream",
    "Specialisation",
    "Hostel",
    "HistoryOfBacklogs",

    #"SGPA_Sem1", # Removed as not in dataset
    #"SGPA_Sem2", # Removed as not in dataset
    #"SGPA_Sem3", # Removed as not in dataset
    #"SGPA_Sem4", # Removed as not in dataset
    #"SGPA_Sem5", # Removed as not in dataset
    #"SGPA_Sem6", # Removed as not in dataset
    #"SGPA_Sem7", # Removed as not in dataset
    #"SGPA_Sem8", # Removed as not in dataset

    "CGPA",
    "AttendancePercent",

    "Internships",
    "Projects",
    "Workshops",
    "Certifications",
    "Publications",

    "AptitudeTestScore",
    "SoftSkillsRating",
    "CodingTestScore",
    "MockInterviewScore",
    "ExtraCurricular"
]

#X = df[features].copy()
X = df[features]

# ============================================================
# 4. CATEGORICAL VARIABLES
# ============================================================

categorical_features = [
    "Gender",
    "City",
    "CollegeTier",
    "Stream",
    "Specialisation",
    "Hostel",
    "HistoryOfBacklogs",
    "ExtraCurricular" # Moved from numerical_features
]


# ============================================================
# 5. NUMERICAL VARIABLES
# ============================================================

numerical_features = [
    #"SGPA_Sem1", # Removed as not in dataset
    #"SGPA_Sem2", # Removed as not in dataset
    #"SGPA_Sem3", # Removed as not in dataset
    #"SGPA_Sem4", # Removed as not in dataset
    #"SGPA_Sem5", # Removed as not in dataset
    #"SGPA_Sem6", # Removed as not in dataset
    #"SGPA_Sem7", # Removed as not in dataset
    #"SGPA_Sem8", # Removed as not in dataset

    "CGPA",
    "AttendancePercent",

    "Internships",
    "Projects",
    "Workshops",
    "Certifications",
    "Publications",

    "AptitudeTestScore",
    "SoftSkillsRating",
    "CodingTestScore",
    "MockInterviewScore"
    #"ExtraCurricular" # Moved to categorical_features
]


# ============================================================
# 6. HANDLE MISSING VALUES
# ============================================================

for col in numerical_features:
    X[col] = X[col].fillna(X[col].median())

for col in categorical_features:
    X[col] = X[col].fillna(X[col].mode()[0])


# ============================================================
# 7. TRAIN-TEST SPLIT
# ============================================================
#stratify=y = Keep the same proportion of values in both training and testing splits.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


# ============================================================
# 8. PREPROCESSING
# ============================================================
#ColumnTransformer allows us to preprocess different types of features differently and combine the results into a single feature matrix

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),

        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            ),
            categorical_features
        )
    ]
)


# ============================================================
# 9. BINOMIAL LOGISTIC REGRESSION
# ============================================================
#The purpose of Pipeline is to connect multiple machine-learning steps into a single sequential workflow.

l1_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                penalty="l1",
                solver="liblinear",
                C=1.0,
                max_iter=2000,
                random_state=42
            )
        )
    ]
)

l1_model.fit(X_train, y_train)

y_pred_l1 = l1_model.predict(X_test)

y_prob_l1 = l1_model.predict_proba(X_test)[:, 1]


# ============================================================
# 10. TRAIN MODEL
# ============================================================

#logistic_model.fit(X_train, y_train) # This line was commented out in the user's code. Keeping it commented.


# ============================================================
# 11. PREDICTION
# ============================================================

#y_pred = logistic_model.predict(X_test) # This line was commented out in the user's code. Keeping it commented.

# Probability of Placement = class 1
#y_probability = logistic_model.predict_proba(X_test)[:, 1] # This line was commented out in the user's code. Keeping it commented.


# ============================================================
# 12. MODEL EVALUATION
# ============================================================

accuracy = accuracy_score(y_test, y_pred_l1)

print("\n===========================================")
print("BINOMIAL LOGISTIC REGRESSION RESULTS")
print("===========================================")

print("\nAccuracy:")
print(round(accuracy, 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_l1))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred_l1,
        target_names=["Not Placed", "Placed"]
    )
)

print("\nROC-AUC:")
print(round(roc_auc_score(y_test, y_prob_l1), 4))

Dataset Shape: (50000, 21)

Columns:
['Gender', 'City', 'CollegeTier', 'Stream', 'Specialisation', 'Hostel', 'HistoryOfBacklogs', 'CGPA', 'AttendancePercent', 'Internships', 'Projects', 'Workshops', 'Certifications', 'Publications', 'AptitudeTestScore', 'SoftSkillsRating', 'CodingTestScore', 'MockInterviewScore', 'ExtraCurricular', 'PlacementStatus', 'IsAnomaly']


/tmp/ipykernel_2885/2442060758.py:132: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[col] = X[col].fillna(X[col].median())
/tmp/ipykernel_2885/2442060758.py:135: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[col] = X[col].fillna(X[col].mode()[0])



BINOMIAL LOGISTIC REGRESSION RESULTS

Accuracy:
0.7903

Confusion Matrix:
[[4229 1021]
 [1076 3674]]

Classification Report:
              precision    recall  f1-score   support

  Not Placed       0.80      0.81      0.80      5250
      Placed       0.78      0.77      0.78      4750

    accuracy                           0.79     10000
   macro avg       0.79      0.79      0.79     10000
weighted avg       0.79      0.79      0.79     10000


ROC-AUC:
0.8769


In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

# ============================================================
# 1. LOAD DATASET
# ============================================================

df = pd.read_csv("/content/placement_predict_50k_adjusted (1).csv")

print("Dataset Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())


# ============================================================
# 2. TARGET VARIABLE
# ============================================================

# PlacementStatus is already:
# 0 = Not Placed
# 1 = Placed

y = df["PlacementStatus"]


# ============================================================
# 3. SELECT PREDICTOR VARIABLES
# ============================================================

features = [
    "Gender",
    "City",
    "CollegeTier",
    "Stream",
    "Specialisation",
    "Hostel",
    "HistoryOfBacklogs",

    #"SGPA_Sem1", # Removed as not in dataset
    #"SGPA_Sem2", # Removed as not in dataset
    #"SGPA_Sem3", # Removed as not in dataset
    #"SGPA_Sem4", # Removed as not in dataset
    #"SGPA_Sem5", # Removed as not in dataset
    #"SGPA_Sem6", # Removed as not in dataset
    #"SGPA_Sem7", # Removed as not in dataset
    #"SGPA_Sem8", # Removed as not in dataset

    "CGPA",
    "AttendancePercent",

    "Internships",
    "Projects",
    "Workshops",
    "Certifications",
    "Publications",

    "AptitudeTestScore",
    "SoftSkillsRating",
    "CodingTestScore",
    "MockInterviewScore",
    "ExtraCurricular"
]

#X = df[features].copy()
X = df[features]

# ============================================================
# 4. CATEGORICAL VARIABLES
# ============================================================

categorical_features = [
    "Gender",
    "City",
    "CollegeTier",
    "Stream",
    "Specialisation",
    "Hostel",
    "HistoryOfBacklogs",
    "ExtraCurricular" # Moved from numerical_features
]


# ============================================================
# 5. NUMERICAL VARIABLES
# ============================================================

numerical_features = [
    #"SGPA_Sem1", # Removed as not in dataset
    #"SGPA_Sem2", # Removed as not in dataset
    #"SGPA_Sem3", # Removed as not in dataset
    #"SGPA_Sem4", # Removed as not in dataset
    #"SGPA_Sem5", # Removed as not in dataset
    #"SGPA_Sem6", # Removed as not in dataset
    #"SGPA_Sem7", # Removed as not in dataset
    #"SGPA_Sem8", # Removed as not in dataset

    "CGPA",
    "AttendancePercent",

    "Internships",
    "Projects",
    "Workshops",
    "Certifications",
    "Publications",

    "AptitudeTestScore",
    "SoftSkillsRating",
    "CodingTestScore",
    "MockInterviewScore"
    #"ExtraCurricular" # Moved to categorical_features
]


# ============================================================
# 6. HANDLE MISSING VALUES
# ============================================================

for col in numerical_features:
    X[col] = X[col].fillna(X[col].median())

for col in categorical_features:
    X[col] = X[col].fillna(X[col].mode()[0])


# ============================================================
# 7. TRAIN-TEST SPLIT
# ============================================================
#stratify=y = Keep the same proportion of values in both training and testing splits.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


# ============================================================
# 8. PREPROCESSING
# ============================================================
#ColumnTransformer allows us to preprocess different types of features differently and combine the results into a single feature matrix

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),

        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first"
            ),
            categorical_features
        )
    ]
)


# ============================================================
# 9. BINOMIAL LOGISTIC REGRESSION
# ============================================================
#The purpose of Pipeline is to connect multiple machine-learning steps into a single sequential workflow.

l2_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                penalty="l2",
                solver="lbfgs",
                C=1.0,
                max_iter=2000,
                random_state=42
            )
        )
    ]
)

l2_model.fit(X_train, y_train)

y_pred_l2 = l2_model.predict(X_test)

y_prob_l2 = l2_model.predict_proba(X_test)[:, 1]


# ============================================================
# 10. TRAIN MODEL
# ============================================================

#logistic_model.fit(X_train, y_train)


# ============================================================
# 11. PREDICTION
# ============================================================

#y_pred = logistic_model.predict(X_test)

# Probability of Placement = class 1
#y_probability = logistic_model.predict_proba(X_test)[:, 1]


# ============================================================
# 12. MODEL EVALUATION
# ============================================================

accuracy = accuracy_score(y_test, y_pred_l2)

print("\n===========================================")
print("BINOMIAL LOGISTIC REGRESSION RESULTS")
print("===========================================")

print("\nAccuracy:")
print(round(accuracy, 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_l2))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred_l2,
        target_names=["Not Placed", "Placed"]
    )
)

print("\nROC-AUC:")
print(round(roc_auc_score(y_test, y_prob_l2), 4))

Dataset Shape: (50000, 21)

Columns:
['Gender', 'City', 'CollegeTier', 'Stream', 'Specialisation', 'Hostel', 'HistoryOfBacklogs', 'CGPA', 'AttendancePercent', 'Internships', 'Projects', 'Workshops', 'Certifications', 'Publications', 'AptitudeTestScore', 'SoftSkillsRating', 'CodingTestScore', 'MockInterviewScore', 'ExtraCurricular', 'PlacementStatus', 'IsAnomaly']


/tmp/ipykernel_2885/1437014852.py:132: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[col] = X[col].fillna(X[col].median())
/tmp/ipykernel_2885/1437014852.py:135: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[col] = X[col].fillna(X[col].mode()[0])



BINOMIAL LOGISTIC REGRESSION RESULTS

Accuracy:
0.7899

Confusion Matrix:
[[4228 1022]
 [1079 3671]]

Classification Report:
              precision    recall  f1-score   support

  Not Placed       0.80      0.81      0.80      5250
      Placed       0.78      0.77      0.78      4750

    accuracy                           0.79     10000
   macro avg       0.79      0.79      0.79     10000
weighted avg       0.79      0.79      0.79     10000


ROC-AUC:
0.8769
